####Google Drive Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Replace the string below with the path you copied
file_path = '/content/drive/MyDrive/Beijing PM2.5/imputed_data/Wanliu.csv'

df = pd.read_csv(file_path, parse_dates=['observation_timestamp'])
df = df.set_index('observation_timestamp').sort_index()


In [3]:
df.head(5)

,current_PM2_5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,PM2_5_next_hour
observation_timestamp,,,,,,,,,,,,,
2013-03-01 00:00:00,8.0,8.0,6.0,28.0,400.0,52.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,9.0
2013-03-01 01:00:00,9.0,9.0,6.0,28.0,400.0,50.0,-1.1,1023.2,-18.2,0.0,N,4.7,3.0
2013-03-01 02:00:00,3.0,6.0,7.0,19.0,400.0,55.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,11.0
2013-03-01 03:00:00,11.0,30.0,8.0,14.0,350.0,54.5,-1.4,1024.5,-19.4,0.0,NW,3.1,3.0
2013-03-01 04:00:00,3.0,13.0,9.0,15.5,300.0,54.0,-2.0,1025.2,-19.5,0.0,N,2.0,3.0


In [ ]:
df.tail(5)

,current_PM2_5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,PM2_5_next_hour
observation_timestamp,,,,,,,,,,,,,
2016-08-31 18:00:00,9.0,39.0,2.0,27.0,200.0,75.0,30.5,988.7,0.0,0.0,NNE,2.0,14.0
2016-08-31 19:00:00,14.0,30.0,2.0,40.0,300.0,58.0,28.6,989.2,-1.5,0.0,NNE,2.1,9.0
2016-08-31 20:00:00,9.0,30.0,2.0,37.0,300.0,56.0,27.7,990.0,-2.2,0.0,NNE,2.3,3.0
2016-08-31 21:00:00,3.0,26.0,2.0,29.0,200.0,63.0,27.9,990.3,-3.1,0.0,N,2.7,10.0
2016-08-31 22:00:00,10.0,42.0,2.0,29.0,200.0,60.0,24.9,990.1,2.9,0.0,N,0.8,5.0


In [ ]:
len(df)

30393

####Feature Engineering

In [25]:
def build_forecast_features(
    df,
    pollutant_cols=['PM10', 'SO2', 'NO2', 'CO', 'O3'],
    weather_cols=['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM'],
    lags=[1, 2, 3, 6, 12, 24],
    rolling_windows=[3, 6, 12, 24]
):
    df = df.copy()

    for raw_col in ['year', 'month', 'day', 'hour', 'current_PM2_5']:
        if raw_col in df.columns:
            df = df.drop(columns=[raw_col])

    all_numeric_cols = pollutant_cols + weather_cols
    new_cols = {}

    # 1. Lag features
    for col in all_numeric_cols:
        for lag in lags:
            new_cols[f'{col}_lag{lag}'] = df[col].shift(lag)

    # 2. Rolling statistics
    for col in all_numeric_cols:
        shifted = df[col].shift(1)
        for window in rolling_windows:
            new_cols[f'{col}_roll{window}_mean'] = shifted.rolling(window).mean()
            new_cols[f'{col}_roll{window}_std']  = shifted.rolling(window).std()
            new_cols[f'{col}_roll{window}_min']  = shifted.rolling(window).min()
            new_cols[f'{col}_roll{window}_max']  = shifted.rolling(window).max()

    # 3. Rate of change / momentum features
    # NOTE: pct_change skipped for RAIN (mostly zero -> undefined % change)
    # Excluded: columns that can be zero/negative, making pct_change undefined or meaningless
    pct_change_exclude = ['RAIN', 'WSPM', 'TEMP', 'DEWP']
    for col in all_numeric_cols:
        new_cols[f'{col}_diff1'] = df[col].diff(1)
        new_cols[f'{col}_diff3'] = df[col].diff(3)
        if col not in pct_change_exclude:
            new_cols[f'{col}_pct_change1'] = df[col].pct_change(1)

    # 4. Cyclical + raw time features
    new_cols['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    new_cols['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    new_cols['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    new_cols['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    new_cols['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
    new_cols['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
    new_cols['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
    new_cols['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365)
    new_cols['hour'] = df.index.hour
    new_cols['dayofweek'] = df.index.dayofweek
    new_cols['month'] = df.index.month
    new_cols['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

    # 5. Wind direction
    wind_dir_map = {
        'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5,
        'E': 90, 'ESE': 112.5, 'SE': 135, 'SSE': 157.5,
        'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5,
        'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5
    }
    if 'wd' in df.columns:
        wd_degrees = df['wd'].map(wind_dir_map)
        new_cols['wd_sin'] = np.sin(np.deg2rad(wd_degrees))
        new_cols['wd_cos'] = np.cos(np.deg2rad(wd_degrees))

    # 6. Interaction features
    if 'TEMP' in df.columns and 'DEWP' in df.columns:
        new_cols['temp_dewp_diff'] = df['TEMP'] - df['DEWP']

    if 'WSPM' in df.columns:
        new_cols['low_wind_flag'] = (df['WSPM'] < df['WSPM'].quantile(0.25)).astype(int)

    # 7. Pollutant interaction features (NEW)
    # Documentation: NO2 and O3 have an inverse relationship in the atmosphere —
    # NO2 destroys ozone via a photochemical reaction, so their ratio captures
    # a real chemical relationship a model can't easily learn from raw values alone.
    if 'NO2' in df.columns and 'O3' in df.columns:
        new_cols['NO2_O3_ratio'] = df['NO2'] / (df['O3'] + 1)  # +1 avoids divide-by-zero

    # Documentation: wind speed affects how much pollution disperses, but its
    # effect depends on how much pollution (PM10, as a proxy) was already present
    # an hour ago — this interaction term lets the model learn that relationship directly.
    if 'PM10' in df.columns and 'WSPM' in df.columns:
        new_cols['PM10_lag1_x_WSPM'] = df['PM10'].shift(1) * df['WSPM']

    new_cols_df = pd.DataFrame(new_cols, index=df.index)
    df = pd.concat([df, new_cols_df], axis=1)

    if 'wd' in df.columns:
        df = df.drop(columns=['wd'])

    return df

featured_df = build_forecast_features(df)
featured_df = featured_df.dropna()
print(featured_df.shape)

(30340, 275)


#### ML Models (exploratory single-station check)

In [26]:
import numpy as np
import pandas as pd

# Target and features
target_col = 'PM2_5_next_hour'
drop_cols = [target_col]  # only exclude the target itself — everything else is a feature

feature_cols = [c for c in featured_df.columns if c not in drop_cols]

X = featured_df[feature_cols]
y = featured_df[target_col]

print(f"Features: {X.shape[1]}, Rows: {X.shape[0]}")

Features: 274, Rows: 30340


In [27]:
import numpy as np

# Replace inf/-inf with NaN everywhere in X
X = X.replace([np.inf, -np.inf], np.nan)

# Check how many rows are affected
print("Rows with inf (before replace):", np.isinf(featured_df[feature_cols]).any(axis=1).sum())
print("Rows with NaN after replace:", X.isna().any(axis=1).sum())

# Drop rows with any remaining NaN (from the inf replacement)
mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

print(f"Final shape after cleaning: {X.shape}")

Rows with inf (before replace): 0
Rows with NaN after replace: 0
Final shape after cleaning: (30340, 274)


In [20]:
from sklearn.model_selection import TimeSeriesSplit

# n_splits = number of train/test folds, each fold's test set comes AFTER its train set chronologically
tscv = TimeSeriesSplit(n_splits=5)

# Quick sanity check — confirm folds are chronological and non-overlapping
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    train_dates = X.index[train_idx]
    test_dates = X.index[test_idx]
    print(f"Fold {fold}: train {train_dates.min()} → {train_dates.max()} "
          f"({len(train_idx)} rows) | test {test_dates.min()} → {test_dates.max()} ({len(test_idx)} rows)")

Fold 0: train 2013-03-02 00:00:00 → 2013-09-29 09:00:00 (5060 rows) | test 2013-09-29 10:00:00 → 2014-04-29 16:00:00 (5056 rows)
Fold 1: train 2013-03-02 00:00:00 → 2014-04-29 16:00:00 (10116 rows) | test 2014-04-29 17:00:00 → 2014-11-28 15:00:00 (5056 rows)
Fold 2: train 2013-03-02 00:00:00 → 2014-11-28 15:00:00 (15172 rows) | test 2014-11-28 16:00:00 → 2015-06-30 19:00:00 (5056 rows)
Fold 3: train 2013-03-02 00:00:00 → 2015-06-30 19:00:00 (20228 rows) | test 2015-06-30 20:00:00 → 2016-01-31 04:00:00 (5056 rows)
Fold 4: train 2013-03-02 00:00:00 → 2016-01-31 04:00:00 (25284 rows) | test 2016-01-31 05:00:00 → 2016-08-31 22:00:00 (5056 rows)


# Global Model Training & Validation

In [30]:
# ============================================================
# Single Global Model Across All Stations
# ============================================================
# Goal: combine all 12 stations into one dataset, with 'station' as
# a feature, so LightGBM has ~10x more data to learn shared
# pollution-weather patterns from, while still specializing per station.

import glob, os

station_csv_folder = "/content/drive/MyDrive/Beijing PM2.5/imputed_data"
station_files = glob.glob(os.path.join(station_csv_folder, "*.csv"))

all_station_dfs = []
for csv_path in station_files:
    station_name = os.path.basename(csv_path).replace(".csv", "")
    sdf = pd.read_csv(csv_path, parse_dates=['observation_timestamp']).set_index('observation_timestamp').sort_index()
    sfeat = build_forecast_features(sdf)
    sfeat['station'] = station_name
    all_station_dfs.append(sfeat)

global_df = pd.concat(all_station_dfs, axis=0)
global_df = global_df.replace([np.inf, -np.inf], np.nan).dropna()
global_df['station'] = global_df['station'].astype('category')
global_df = global_df.sort_index()  # keep chronological order for TimeSeriesSplit

X_global = global_df[[c for c in global_df.columns if c != target_col]]
y_global = global_df[target_col]

print(f"Global dataset: {X_global.shape[0]} rows, {X_global.shape[1]} features")

# ---- Evaluate across folds ----
global_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_global)):
    X_train, X_test = X_global.iloc[train_idx], X_global.iloc[test_idx]
    y_train, y_test = y_global.iloc[train_idx], y_global.iloc[test_idx]

    model = LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.02,
        max_depth=12,
        num_leaves=90,
        subsample=0.8,           # regularization — randomly uses 80% of rows per tree, reduces overfitting
        colsample_bytree=0.8,    # regularization — randomly uses 80% of features per tree
        min_child_samples=20,    # regularization — requires at least 20 rows per leaf, prevents overly specific splits
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_train, y_train, categorical_feature=['station'])
    y_pred = model.predict(X_test)

    global_results.append({
        'fold': fold,
        'R2': r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })
    print(f"Fold {fold}: R2={global_results[-1]['R2']:.4f}, RMSE={global_results[-1]['RMSE']:.3f}")

global_results_df = pd.DataFrame(global_results)
print(f"\nMean R2: {global_results_df['R2'].mean():.4f}, Mean RMSE: {global_results_df['RMSE'].mean():.3f}")

# ---- Train final model on ALL data ----
final_global_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    max_depth=12,
    num_leaves=90,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
final_global_model.fit(X_global, y_global, categorical_feature=['station'])

global_feature_cols = X_global.columns.tolist()
print("\n✅ Final global model trained on all stations")

Global dataset: 359873 rows, 275 features
Fold 0: R2=0.8476, RMSE=35.736
Fold 1: R2=0.8993, RMSE=22.296
Fold 2: R2=0.8877, RMSE=24.512
Fold 3: R2=0.9134, RMSE=27.268
Fold 4: R2=0.8540, RMSE=23.650

Mean R2: 0.8804, Mean RMSE: 26.692

✅ Final global model trained on all stations


# Predict and Build submission file

In [31]:
# ============================================================
# Predict using the cleaned test file
# ============================================================
train_folder = "/content/drive/MyDrive/Beijing PM2.5/imputed_data"
test_df = pd.read_csv('/content/drive/MyDrive/Beijing PM2.5/test_cleaned.csv', parse_dates=['observation_timestamp'])

max_lookback = 24
all_predictions = []

for station_name in test_df['station'].unique():
    print(f"Predicting: {station_name}")

    train_hist = pd.read_csv(f"{train_folder}/{station_name}.csv",
                              parse_dates=['observation_timestamp']).set_index('observation_timestamp').sort_index()

    test_station = test_df[test_df['station'] == station_name].set_index('observation_timestamp').sort_index()
    if test_station.empty:
        continue

    # Stitch train tail onto test for lookback (test is already clean, no ffill/bfill needed)
    combined = pd.concat([train_hist.tail(max_lookback * 2), test_station.drop(columns=['id'])], axis=0)
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()

    combined_featured = build_forecast_features(combined)
    combined_featured = combined_featured.replace([np.inf, -np.inf], np.nan)
    combined_featured['station'] = station_name

    test_featured = combined_featured.loc[combined_featured.index.isin(test_station.index)]
    X_test_final = test_featured.reindex(columns=global_feature_cols)
    X_test_final['station'] = X_test_final['station'].astype('category')

    nan_rows = X_test_final.isna().any(axis=1).sum()
    if nan_rows > 0:
        print(f"  ⚠️ {nan_rows} rows with NaN — filling with 0")
        non_category_cols = X_test_final.select_dtypes(exclude='category').columns
        X_test_final[non_category_cols] = X_test_final[non_category_cols].fillna(0)

    preds = final_global_model.predict(X_test_final)

    ids = test_station.loc[X_test_final.index, 'id'].values
    all_predictions.append(pd.DataFrame({'id': ids, 'PM2_5_next_hour': preds}))
    print(f"  ✅ {len(ids)} predictions")

submission_df = pd.concat(all_predictions, ignore_index=True)
submission_df['PM2_5_next_hour'] = submission_df['PM2_5_next_hour'].clip(lower=0)

print(f"\nTotal: {submission_df.shape[0]} (expected: {test_df.shape[0]})")
submission_path = "/content/drive/MyDrive/Beijing PM2.5/submission_v1.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Saved: {submission_path}")

Predicting: Aotizhongxin
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4292 predictions
Predicting: Changping
  ⚠️ 59 rows with NaN — filling with 0
  ✅ 4297 predictions
Predicting: Dingling
  ⚠️ 57 rows with NaN — filling with 0
  ✅ 4143 predictions
Predicting: Dongsi
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4234 predictions
Predicting: Guanyuan
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4248 predictions
Predicting: Gucheng
  ⚠️ 37 rows with NaN — filling with 0
  ✅ 4251 predictions
Predicting: Huairou
  ⚠️ 124 rows with NaN — filling with 0
  ✅ 4280 predictions
Predicting: Nongzhanguan
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4260 predictions
Predicting: Shunyi
  ⚠️ 237 rows with NaN — filling with 0
  ✅ 4210 predictions
Predicting: Tiantan
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4284 predictions
Predicting: Wanliu
  ⚠️ 94 rows with NaN — filling with 0
  ✅ 4288 predictions
Predicting: Wanshouxigong
  ⚠️ 66 rows with NaN — filling with 0
  ✅ 4276 predictions

Total: 51063 (expect